In [6]:
! pip install -q python-terrier beir

In [9]:
import pyterrier as pt
if not pt.started():
    pt.init()

from beir import util
from beir.datasets.data_loader import GenericDataLoader
import nltk
from tqdm import tqdm
from nltk.stem import LancasterStemmer
import pandas as pd
import os
import re
import json
import numpy as np

### rm3 retriever class for beir
class RM3Retriever:
    def __init__(self, corpus, index_path="./index"):
        """Initialize RM3 retriever with BEIR corpus"""
        self.index_path = index_path
        self.corpus = corpus
        nltk.download('punkt', quiet=True)
        self.lancaster_stemmer = LancasterStemmer()
        self._build_index()

    def stemmer(self, text):
        """
        Lancaster stemming algorithm - known for aggressive stemming
        that can hurt retrieval performance.

        Justification: Evaluating RM3 with aggressive morphological
        conflation (Lancaster algorithm) vs. standard Porter stemming.
        """
        words = text.lower().split()
        stemmed = [self.lancaster_stemmer.stem(word) for word in words]
        return ' '.join(stemmed)

    def _build_index(self):
        """Build PyTerrier index from BEIR corpus"""
        print(f"Building index with {len(self.corpus)} documents...")

        ### convert beir corpus to pyterrier format
        docs = []
        for doc_id, doc in tqdm(self.corpus.items()):
            text = doc.get('title', '') + ' ' + doc.get('text', '')
            # text = self.stemmer(text) # worse results
            docs.append({'docno': doc_id, 'text': text})

        df = pd.DataFrame(docs)

        ### create index with proper tokenization
        indexer = pt.IterDictIndexer(
            self.index_path,
            overwrite=True,
            meta={'docno': 20, 'text': 4096},
            properties={"termpipelines": "Stopwords"}
        )
        index_ref = indexer.index(df.to_dict('records'))
        self.index = pt.IndexFactory.of(index_ref)
        print(f"Index built successfully at {self.index_path}")

    def clean_query(self, query):
        """Clean query text to be safe for TerrierQL parser"""
        ### keep alphanumeric and basic spaces, remove special chars
        query = re.sub(r'[^\w\s-]', ' ', query)
        ### remove extra whitespace
        query = re.sub(r'\s+', ' ', query).strip()
        ### ensure query is not empty
        if not query:
            query = "query"
        # query = self.stemmer(query) # worse results
        return query

    def search(self, queries, top_k=1000, fb_terms=10, fb_docs=3):
        """
        Search using RM3 query expansion

        Args:
            queries: Dictionary of query_id -> query_text
            top_k: Number of results to return per query
            fb_terms: Number of feedback terms for RM3
            fb_docs: Number of feedback documents for RM3
        """
        print(f"Searching with RM3 (fb_terms={fb_terms}, fb_docs={fb_docs})...")

        ### convert queries to pyterrier format and clean them
        query_list = [
            {'qid': str(qid), 'query': self.clean_query(qtext)}
            for qid, qtext in queries.items()
        ]
        queries_df = pd.DataFrame(query_list)

        ### create standard bm25 retriever
        bm25 = pt.terrier.Retriever(
            self.index,
            wmodel="BM25",
            properties={"termpipelines": "Stopwords"},
            num_results=top_k
        )

        ### create rm3 query expansion
        rm3 = pt.rewrite.RM3(self.index, fb_terms=fb_terms, fb_docs=fb_docs)

        ### pipeline: bm25 -> rm3 expansion -> bm25 with expanded query
        pipeline = bm25 >> rm3 >> bm25

        ### run retrieval
        results = pipeline.transform(queries_df)

        ### convert to beir format: {query_id: {doc_id: score}}
        beir_results = {}
        for qid in queries.keys():
            beir_results[str(qid)] = {}
            query_results = results[results['qid'] == str(qid)]
            for _, row in query_results.iterrows():
                beir_results[str(qid)][row['docno']] = float(row['score'])

        return beir_results

def compute_per_query_metrics(qrels, results, k_values=[10]):
    """Compute per-query NDCG, Recall, and MAP"""
    from beir.retrieval.evaluation import EvaluateRetrieval

    per_query_metrics = {}

    for query_id in qrels.keys():
        per_query_metrics[query_id] = {}

        ### get relevant docs for this query
        relevant_docs = {doc_id: score for doc_id, score in qrels[query_id].items() if score > 0}

        if not relevant_docs or query_id not in results:
            ### no relevant docs or no results for this query
            for k in k_values:
                per_query_metrics[query_id][f'ndcg@{k}'] = 0.0
                per_query_metrics[query_id][f'recall@{k}'] = 0.0
                per_query_metrics[query_id][f'map@{k}'] = 0.0
            continue

        ### get retrieved docs for this query
        retrieved_docs = results[query_id]

        ### sort by score
        sorted_docs = sorted(retrieved_docs.items(), key=lambda x: x[1], reverse=True)

        for k in k_values:
            top_k_docs = sorted_docs[:k]

            ### compute ndcg@k
            dcg = 0.0
            for i, (doc_id, score) in enumerate(top_k_docs):
                rel = relevant_docs.get(doc_id, 0)
                dcg += rel / np.log2(i + 2)  # i+2 because i starts at 0

            ### compute ideal dcg
            ideal_scores = sorted(relevant_docs.values(), reverse=True)
            idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_scores[:k]))

            ndcg = dcg / idcg if idcg > 0 else 0.0
            per_query_metrics[query_id][f'ndcg@{k}'] = ndcg

            ### compute recall@k
            retrieved_relevant = set([doc_id for doc_id, _ in top_k_docs]) & set(relevant_docs.keys())
            recall = len(retrieved_relevant) / len(relevant_docs) if len(relevant_docs) > 0 else 0.0
            per_query_metrics[query_id][f'recall@{k}'] = recall

            ### compute map@k (average precision at k)
            ap = 0.0
            num_relevant_seen = 0
            for i, (doc_id, score) in enumerate(top_k_docs):
                if doc_id in relevant_docs:
                    num_relevant_seen += 1
                    precision_at_i = num_relevant_seen / (i + 1)
                    ap += precision_at_i

            ap = ap / min(len(relevant_docs), k) if len(relevant_docs) > 0 else 0.0
            per_query_metrics[query_id][f'map@{k}'] = ap

    return per_query_metrics

def evaluate_beir(results, qrels, k_values=[1, 3, 5, 10, 100, 1000]):
    """Evaluate results using BEIR metrics"""
    from beir.retrieval.evaluation import EvaluateRetrieval

    ### convert qrels keys to strings to match results
    qrels_str = {str(k): v for k, v in qrels.items()}

    ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels_str, results, k_values)

    ### compute per-query metrics
    per_query_metrics = compute_per_query_metrics(qrels_str, results, k_values=[10])

    ### print results
    print("\nResults:")
    print(f"NDCG@10: {ndcg['NDCG@10']:.4f}")
    print(f"NDCG@100: {ndcg['NDCG@100']:.4f}")
    print(f"MAP@10: {_map['MAP@10']:.4f}")
    print(f"Recall@10: {recall['Recall@10']:.4f}")
    print(f"Recall@100: {recall['Recall@100']:.4f}")

    return ndcg, _map, recall, precision, per_query_metrics

### main evaluation function
def run_rm3_on_beir(dataset_name, fb_terms=10, fb_docs=3):
    """Run RM3 on a BEIR dataset"""
    print(f"\n{'='*60}")
    print(f"Running RM3 on {dataset_name}")
    print(f"{'='*60}")

    ### download and load dataset
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"
    out_dir = "datasets"
    data_path = util.download_and_unzip(url, out_dir)

    corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

    print(f"Loaded {len(corpus)} documents, {len(queries)} queries")

    ### initialize rm3 retriever
    index_path = f"./index_{dataset_name}"
    retriever = RM3Retriever(corpus, index_path=index_path)

    ### run search
    results = retriever.search(queries, top_k=1000, fb_terms=fb_terms, fb_docs=fb_docs)

    ### evaluate
    metrics = evaluate_beir(results, qrels)

    return results, metrics

### run on all three datasets
datasets = ["nfcorpus", "scifact", "trec-covid"]
all_results = {}
per_query_metrics = {}

for dataset in datasets:
    try:
        results, metrics = run_rm3_on_beir(dataset, fb_terms=10, fb_docs=3)
        ndcg, _map, recall, precision, per_query = metrics

        all_results[dataset] = {
            'results': results,
            'metrics': (ndcg, _map, recall, precision)
        }

        ### store per-query metrics for this dataset
        per_query_metrics[dataset] = per_query

    except Exception as e:
        print(f"\nError processing {dataset}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

### summary table
print("\n" + "="*60)
print("SUMMARY: RM3 Results on BEIR Datasets")
print("="*60)
print(f"{'Dataset':<15} {'NDCG@10':<10} {'NDCG@100':<10} {'Recall@100':<12}")
print("-"*60)
for dataset in datasets:
    if dataset in all_results:
        ndcg = all_results[dataset]['metrics'][0]
        recall = all_results[dataset]['metrics'][2]
        print(f"{dataset:<15} {ndcg['NDCG@10']:<10.4f} {ndcg['NDCG@100']:<10.4f} {recall['Recall@100']:<12.4f}")
    else:
        print(f"{dataset:<15} {'FAILED':<10} {'FAILED':<10} {'FAILED':<12}")

### save per-query metrics to json
with open('rm3_per_query_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(per_query_metrics, f, indent=2, ensure_ascii=False)

print("\nPer-query metrics saved to: rm3_per_query_metrics.json")


Running RM3 on nfcorpus


/tmp/ipython-input-3838093954.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


  0%|          | 0/3633 [00:00<?, ?it/s]

Loaded 3633 documents, 323 queries
Building index with 3633 documents...


100%|██████████| 3633/3633 [00:00<00:00, 470654.39it/s]


...

Index built successfully at ./index_scifact
Searching with RM3 (fb_terms=10, fb_docs=3)...

Results:
NDCG@10: 0.6109
NDCG@100: 0.6313
MAP@10: 0.5491
Recall@10: 0.7873
Recall@100: 0.8799

Running RM3 on trec-covid


  0%|          | 0/171332 [00:00<?, ?it/s]

Loaded 171332 documents, 50 queries
Building index with 171332 documents...


100%|██████████| 171332/171332 [00:00<00:00, 596079.46it/s]


23:41:03.426 [ForkJoinPool-15-worker-3] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
23:42:28.402 [ForkJoinPool-15-worker-3] WARN org.terrier.structures.indexing.Indexer -- Indexed 17 empty documents
Index built successfully at ./index_trec-covid
Searching with RM3 (fb_terms=10, fb_docs=3)...

Results:
NDCG@10: 0.4062
NDCG@100: 0.3771
MAP@10: 0.0085
Recall@10: 0.0123
Recall@100: 0.0949

SUMMARY: RM3 Results on BEIR Datasets
Dataset         NDCG@10    NDCG@100   Recall@100  
------------------------------------------------------------
nfcorpus        0.3259     0.2947     0.2944      
scifact         0.6109     0.6313     0.8799      
trec-covid      0.4062     0.3771     0.0949      

Per-query metrics saved to: rm3_per_query_metrics.json


In [10]:
import pyterrier as pt
if not pt.started():
    pt.init()

from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd
import os
import re
import json
import numpy as np

### rm3 retriever class for beir
class RM3Retriever:
    def __init__(self, corpus, index_path="./index"):
        """Initialize RM3 retriever with BEIR corpus"""
        self.index_path = index_path
        self.corpus = corpus
        self._build_index()

    def _build_index(self):
        """Build PyTerrier index from BEIR corpus"""
        print(f"Building index with {len(self.corpus)} documents...")

        ### convert beir corpus to pyterrier format
        docs = []
        for doc_id, doc in self.corpus.items():
            text = doc.get('title', '') + ' ' + doc.get('text', '')
            docs.append({'docno': doc_id, 'text': text})

        df = pd.DataFrame(docs)

        ### create index with proper tokenization
        indexer = pt.IterDictIndexer(
            self.index_path,
            overwrite=True,
            meta={'docno': 20, 'text': 4096},
            properties={"termpipelines": "Stopwords"}
        )
        index_ref = indexer.index(df.to_dict('records'))
        self.index = pt.IndexFactory.of(index_ref)
        print(f"Index built successfully at {self.index_path}")

    def clean_query(self, query):
        """Clean query text to be safe for TerrierQL parser"""
        ### keep alphanumeric and basic spaces, remove special chars
        query = re.sub(r'[^\w\s-]', ' ', query)
        ### remove extra whitespace
        query = re.sub(r'\s+', ' ', query).strip()
        ### ensure query is not empty
        if not query:
            query = "query"
        return query

    def search(self, queries, top_k=1000, fb_terms=10, fb_docs=3):
        """
        Search using RM3 query expansion

        Args:
            queries: Dictionary of query_id -> query_text
            top_k: Number of results to return per query
            fb_terms: Number of feedback terms for RM3
            fb_docs: Number of feedback documents for RM3
        """
        print(f"Searching with RM3 (fb_terms={fb_terms}, fb_docs={fb_docs})...")

        ### convert queries to pyterrier format and clean them
        query_list = [
            {'qid': str(qid), 'query': self.clean_query(qtext)}
            for qid, qtext in queries.items()
        ]
        queries_df = pd.DataFrame(query_list)

        ### create standard bm25 retriever
        bm25 = pt.terrier.Retriever(
            self.index,
            wmodel="BM25",
            properties={"termpipelines": "Stopwords"},
            num_results=top_k
        )

        ### create rm3 query expansion
        rm3 = pt.rewrite.RM3(self.index, fb_terms=fb_terms, fb_docs=fb_docs)

        ### pipeline: bm25 -> rm3 expansion -> bm25 with expanded query
        pipeline = bm25 >> rm3 >> bm25

        ### run retrieval
        results = pipeline.transform(queries_df)

        ### convert to beir format: {query_id: {doc_id: score}}
        beir_results = {}
        for qid in queries.keys():
            beir_results[str(qid)] = {}
            query_results = results[results['qid'] == str(qid)]
            for _, row in query_results.iterrows():
                beir_results[str(qid)][row['docno']] = float(row['score'])

        return beir_results

def load_paraphrased_queries(dataset_name):
    """Load paraphrased queries from JSON file"""
    json_file = f"{dataset_name}_query_paraphrased_gpt4o.json"

    try:
        with open(json_file, encoding='utf-8') as f:
            queries_para = json.load(f)

        ### extract the paraphrased queries
        queries_p = {}
        for q in queries_para:
            queries_p[q] = queries_para[q]['query_p']

        print(f"Loaded {len(queries_p)} paraphrased queries from {json_file}")
        return queries_p
    except FileNotFoundError:
        print(f"Warning: Paraphrased query file {json_file} not found. Using original queries.")
        return None
    except Exception as e:
        print(f"Error loading paraphrased queries: {str(e)}. Using original queries.")
        return None

def compute_per_query_metrics(qrels, results, k_values=[10]):
    """Compute per-query NDCG, Recall, and MAP"""
    from beir.retrieval.evaluation import EvaluateRetrieval

    per_query_metrics = {}

    for query_id in qrels.keys():
        per_query_metrics[query_id] = {}

        ### get relevant docs for this query
        relevant_docs = {doc_id: score for doc_id, score in qrels[query_id].items() if score > 0}

        if not relevant_docs or query_id not in results:
            ### no relevant docs or no results for this query
            for k in k_values:
                per_query_metrics[query_id][f'ndcg@{k}'] = 0.0
                per_query_metrics[query_id][f'recall@{k}'] = 0.0
                per_query_metrics[query_id][f'map@{k}'] = 0.0
            continue

        ### get retrieved docs for this query
        retrieved_docs = results[query_id]

        ### sort by score
        sorted_docs = sorted(retrieved_docs.items(), key=lambda x: x[1], reverse=True)

        for k in k_values:
            top_k_docs = sorted_docs[:k]

            ### compute ndcg@k
            dcg = 0.0
            for i, (doc_id, score) in enumerate(top_k_docs):
                rel = relevant_docs.get(doc_id, 0)
                dcg += rel / np.log2(i + 2)  # i+2 because i starts at 0

            ### compute ideal dcg
            ideal_scores = sorted(relevant_docs.values(), reverse=True)
            idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_scores[:k]))

            ndcg = dcg / idcg if idcg > 0 else 0.0
            per_query_metrics[query_id][f'ndcg@{k}'] = ndcg

            ### compute recall@k
            retrieved_relevant = set([doc_id for doc_id, _ in top_k_docs]) & set(relevant_docs.keys())
            recall = len(retrieved_relevant) / len(relevant_docs) if len(relevant_docs) > 0 else 0.0
            per_query_metrics[query_id][f'recall@{k}'] = recall

            ### compute map@k (average precision at k)
            ap = 0.0
            num_relevant_seen = 0
            for i, (doc_id, score) in enumerate(top_k_docs):
                if doc_id in relevant_docs:
                    num_relevant_seen += 1
                    precision_at_i = num_relevant_seen / (i + 1)
                    ap += precision_at_i

            ap = ap / min(len(relevant_docs), k) if len(relevant_docs) > 0 else 0.0
            per_query_metrics[query_id][f'map@{k}'] = ap

    return per_query_metrics

def evaluate_beir(results, qrels, k_values=[1, 3, 5, 10, 100, 1000]):
    """Evaluate results using BEIR metrics"""
    from beir.retrieval.evaluation import EvaluateRetrieval

    ### convert qrels keys to strings to match results
    qrels_str = {str(k): v for k, v in qrels.items()}

    ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels_str, results, k_values)

    ### compute per-query metrics
    per_query_metrics = compute_per_query_metrics(qrels_str, results, k_values=[10])

    ### print results
    print("\nResults:")
    print(f"NDCG@10: {ndcg['NDCG@10']:.4f}")
    print(f"NDCG@100: {ndcg['NDCG@100']:.4f}")
    print(f"MAP@10: {_map['MAP@10']:.4f}")
    print(f"Recall@10: {recall['Recall@10']:.4f}")
    print(f"Recall@100: {recall['Recall@100']:.4f}")

    return ndcg, _map, recall, precision, per_query_metrics

### main evaluation function
def run_rm3_on_beir(dataset_name, fb_terms=10, fb_docs=3, use_paraphrased=True):
    """Run RM3 on a BEIR dataset"""
    print(f"\n{'='*60}")
    print(f"Running RM3 on {dataset_name}")
    print(f"{'='*60}")

    ### download and load dataset
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"
    out_dir = "datasets"
    data_path = util.download_and_unzip(url, out_dir)

    corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

    print(f"Loaded {len(corpus)} documents, {len(queries)} queries")

    ### load paraphrased queries if requested
    if use_paraphrased:
        paraphrased_queries = load_paraphrased_queries(dataset_name)
        if paraphrased_queries is not None:
            ### replace queries with paraphrased versions
            queries = paraphrased_queries
            print(f"Using {len(queries)} paraphrased queries")
        else:
            print("Using original queries")
    else:
        print("Using original queries")

    ### initialize rm3 retriever
    index_path = f"./index_{dataset_name}"
    retriever = RM3Retriever(corpus, index_path=index_path)

    ### run search
    results = retriever.search(queries, top_k=1000, fb_terms=fb_terms, fb_docs=fb_docs)

    ### evaluate
    metrics = evaluate_beir(results, qrels)

    return results, metrics

### run on all three datasets
datasets = ["nfcorpus", "scifact", "trec-covid"]
all_results = {}
per_query_metrics = {}

for dataset in datasets:
    try:
        ### run with paraphrased queries
        results, metrics = run_rm3_on_beir(
            dataset,
            fb_terms=10,
            fb_docs=3,
            use_paraphrased=True
        )
        ndcg, _map, recall, precision, per_query = metrics

        all_results[dataset] = {
            'results': results,
            'metrics': (ndcg, _map, recall, precision)
        }

        ### store per-query metrics for this dataset
        per_query_metrics[dataset] = per_query

    except Exception as e:
        print(f"\nError processing {dataset}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

### summary table
print("\n" + "="*60)
print("SUMMARY: RM3 Results on BEIR Datasets (Paraphrased Queries)")
print("="*60)
print(f"{'Dataset':<15} {'NDCG@10':<10} {'NDCG@100':<10} {'Recall@100':<12}")
print("-"*60)
for dataset in datasets:
    if dataset in all_results:
        ndcg = all_results[dataset]['metrics'][0]
        recall = all_results[dataset]['metrics'][2]
        print(f"{dataset:<15} {ndcg['NDCG@10']:<10.4f} {ndcg['NDCG@100']:<10.4f} {recall['Recall@100']:<12.4f}")
    else:
        print(f"{dataset:<15} {'FAILED':<10} {'FAILED':<10} {'FAILED':<12}")

### save per-query metrics to json
with open('rm3_paraphrased_per_query_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(per_query_metrics, f, indent=2, ensure_ascii=False)

print("\nPer-query metrics saved to: rm3_paraphrased_per_query_metrics.json")


Running RM3 on nfcorpus


/tmp/ipython-input-986940524.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


  0%|          | 0/3633 [00:00<?, ?it/s]

Loaded 3633 documents, 323 queries
Loaded 323 paraphrased queries from nfcorpus_query_paraphrased_gpt4o.json
Using 323 paraphrased queries
Building index with 3633 documents...
Index built successfully at ./index_nfcorpus
Searching with RM3 (fb_terms=10, fb_docs=3)...

Results:
NDCG@10: 0.2700
NDCG@100: 0.2497
MAP@10: 0.0966
Recall@10: 0.1300
Recall@100: 0.2633

Running RM3 on scifact


  0%|          | 0/5183 [00:00<?, ?it/s]

Loaded 5183 documents, 300 queries
Loaded 300 paraphrased queries from scifact_query_paraphrased_gpt4o.json
Using 300 paraphrased queries
Building index with 5183 documents...
Index built successfully at ./index_scifact
Searching with RM3 (fb_terms=10, fb_docs=3)...

Results:
NDCG@10: 0.6034
NDCG@100: 0.6308
MAP@10: 0.5478
Recall@10: 0.7616
Recall@100: 0.8850

Running RM3 on trec-covid


  0%|          | 0/171332 [00:00<?, ?it/s]

Loaded 171332 documents, 50 queries
Loaded 50 paraphrased queries from trec-covid_query_paraphrased_gpt4o.json
Using 50 paraphrased queries
Building index with 171332 documents...
23:44:52.947 [ForkJoinPool-18-worker-3] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
23:46:12.756 [ForkJoinPool-18-worker-3] WARN org.terrier.structures.indexing.Indexer -- Indexed 17 empty documents
Index built successfully at ./index_trec-covid
Searching with RM3 (fb_terms=10, fb_docs=3)...

Results:
NDCG@10: 0.3333
NDCG@100: 0.2989
MAP@10: 0.0076
Recall@10: 0.0113
Recall@100: 0.0835

SUMMARY: RM3 Results on BEIR Datasets (Paraphrased Queries)
Dataset         NDCG@10    NDCG@100   Recall@100  
------------------------------------------------------------
nfcorpus        0.2700     0.2497     0.2633      
scifact         0.6034     0.6308     0.8850      
trec-covid      0.3333     0.2989     0.0835      

Per-query metrics 